# NB-05 — めぐ指数: 集計関数バリデーション（今週のめぐ指数 A/B/C）

**目的**: 「今週のめぐ指数」の3集計パターン（A: 最大値 / B: 加重平均 / C: 条件絞り最大値）の着順予測有効性を比較し、パターンBの加重平均ウェイトを最適化する

**評価指標**:
- 1着馬の予測精度（指数1位が1着になる率）
- 3着内（複勝）の予測精度
- 指数デシル別の勝率・複勝率
- スピアマン順位相関（めぐ指数順位 vs 着順）

**実行順序**: NB-02 の実行後（`megu_index_results.parquet` が必要）

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/keiba-vpn')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from scipy import stats
from itertools import product

INPUT_NB01 = Path('output/nb01')
INPUT_NB02 = Path('output/nb02')
OUTPUT_DIR = Path('output/nb05')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_index = pd.read_parquet(INPUT_NB02 / 'megu_index_results.parquet')
df_race  = pd.read_parquet(INPUT_NB01 / 'megu_dataset.parquet')

# レース結果とめぐ指数を結合
merge_cols = ['race_id', 'horse_id', 'finish_pos', 'distance', 'surface',
              'track_condition', 'race_date', 'course']
df = df_index.merge(
    df_race[merge_cols].drop_duplicates(),
    on=['race_id', 'horse_id'], how='inner'
)
df['race_date'] = pd.to_datetime(df['race_date'])

# --- 学習/テスト分割（NB-02 の split 列を使用）---
# split 列がない場合（旧バージョンの出力）は全データを学習データとして扱う
if 'split' not in df.columns:
    print('WARNING: split 列が見つかりません。全データを学習データとして使用します。')
    df['split'] = 'train'

df_train = df[df['split'] == 'train'].copy()
df_test  = df[df['split'] == 'test'].copy()

print(f'統合データ: {len(df):,} 行  ({df["race_id"].nunique():,} レース)')
print(f'  学習: {len(df_train):,} 行  ({df_train["race_id"].nunique():,} レース)')
print(f'  テスト: {len(df_test):,} 行  ({df_test["race_id"].nunique():,} レース)')

## 1. 「今週のめぐ指数」の算出ロジック実装

In [ ]:
# 直近5走のめぐ指数を馬ごとに取得（全データに対して計算）
# ラグは時系列順に計算する必要があるため、全期間のデータを使ってシフト
# → テスト期間のレースも直近5走にトレーニング期間のレースを参照できる

df_sorted = df.sort_values(['horse_id', 'race_date', 'race_id'])

def get_recent_n(group, n=5):
    """各行の時点での直近N走のめぐ指数を返す"""
    indices = []
    for i in range(1, n + 1):
        indices.append(group['megu_index'].shift(i).rename(f'mi_lag{i}'))
    return pd.concat(indices, axis=1)

lag_df = df_sorted.groupby('horse_id', group_keys=False).apply(get_recent_n)
df_with_lags = pd.concat([df_sorted.reset_index(drop=True), lag_df.reset_index(drop=True)], axis=1)

# 学習/テストに分割（lag 計算後に分割する）
df_lags_train = df_with_lags[df_with_lags['split'] == 'train'].copy()
df_lags_test  = df_with_lags[df_with_lags['split'] == 'test'].copy()

print(f'ラグ特徴量追加後: {len(df_with_lags):,} 行')
print(f'  学習: {len(df_lags_train):,} 行  テスト: {len(df_lags_test):,} 行')
print(df_with_lags[['race_id', 'horse_id', 'split', 'megu_index', 'mi_lag1', 'mi_lag2']].head())

In [ ]:
def calc_pattern_a(row):
    """パターンA: 直近5走最大値"""
    vals = [row.get(f'mi_lag{i}') for i in range(1, 6)]
    vals = [v for v in vals if pd.notna(v)]
    return max(vals) if vals else np.nan

def calc_pattern_b(row, weights=(0.35, 0.25, 0.20, 0.12, 0.08)):
    """パターンB: 加重平均（デフォルトウェイト）"""
    vals = [row.get(f'mi_lag{i}') for i in range(1, 6)]
    total_w, total_wv = 0, 0
    for v, w in zip(vals, weights):
        if pd.notna(v):
            total_w  += w
            total_wv += w * v
    return total_wv / total_w if total_w > 0 else np.nan

def calc_pattern_c(row, target_surface=None, target_distance_band=None):
    """パターンC: 直近3走最大値（簡易版）"""
    vals = [row.get(f'mi_lag{i}') for i in range(1, 4)]
    vals = [v for v in vals if pd.notna(v)]
    return max(vals) if vals else np.nan

# 学習データのみでパターン算出（グリッドサーチ用）
df_lags_train['pattern_a'] = df_lags_train.apply(calc_pattern_a, axis=1)
df_lags_train['pattern_b'] = df_lags_train.apply(calc_pattern_b, axis=1)

avail = df_lags_train['pattern_b'].notna().mean()
print(f'パターンB 算出可能率（学習データ）: {avail*100:.1f}%')
print(df_lags_train[['pattern_a', 'pattern_b']].describe())

## 2. 着順予測精度の評価

In [ ]:
def eval_pattern(df, score_col):
    """集計パターンの着順予測精度を評価する"""
    df = df.dropna(subset=[score_col, 'finish_pos']).copy()
    if len(df) < 100:
        return {'n': 0, 'n_races': 0, 'win_top1_rate': np.nan,
                'show_top1_rate': np.nan, 'spearman_corr': np.nan}

    df['score_rank'] = df.groupby('race_id')[score_col].rank(ascending=False, method='min')

    win_1st        = df[df['finish_pos'] == 1]['score_rank'].eq(1).mean()
    show_rate_top  = df[df['score_rank'] == 1]['finish_pos'].le(3).mean()

    spearman_corrs = df.groupby('race_id').apply(
        lambda g: stats.spearmanr(g[score_col], g['finish_pos'])[0] if len(g) >= 3 else np.nan
    )
    mean_spearman = spearman_corrs.mean()

    return {
        'n': len(df),
        'n_races': df['race_id'].nunique(),
        'win_top1_rate': win_1st,
        'show_top1_rate': show_rate_top,
        'spearman_corr': mean_spearman,
    }

# 学習データで各パターンを評価
df_eval_train = df_lags_train[df_lags_train['finish_pos'].notna()].copy()

results = {}
for name, col in [('A (max5)', 'pattern_a'), ('B (weighted_avg)', 'pattern_b'), ('megu_index (raw)', 'megu_index')]:
    results[name] = eval_pattern(df_eval_train, col)

df_eval_result = pd.DataFrame(results).T
print('=== 集計パターン別 着順予測精度（学習データ）===')
print(df_eval_result.to_string(float_format='{:.4f}'.format))

## 3. パターンBのウェイト最適化

In [ ]:
# ウェイトのグリッドサーチ（学習データのみで最適化）
# データリークを防ぐため: 最適ウェイトは学習データで決定、テストデータで検証

best_spearman = -999
best_weights  = (0.35, 0.25, 0.20, 0.12, 0.08)

candidate_weights = [
    (0.25, 0.20, 0.20, 0.20, 0.15),
    (0.30, 0.25, 0.20, 0.15, 0.10),
    (0.35, 0.25, 0.20, 0.12, 0.08),  # デフォルト
    (0.40, 0.25, 0.18, 0.10, 0.07),
    (0.45, 0.25, 0.15, 0.09, 0.06),
    (0.50, 0.25, 0.13, 0.07, 0.05),
    (0.30, 0.30, 0.20, 0.12, 0.08),
    (0.35, 0.30, 0.18, 0.11, 0.06),
]

weight_results = []
for weights in candidate_weights:
    w_sum        = sum(weights)
    weights_norm = tuple(w / w_sum for w in weights)

    df_eval_train['pattern_b_test'] = df_eval_train.apply(
        lambda r: calc_pattern_b(r, weights=weights_norm), axis=1
    )
    r = eval_pattern(df_eval_train, 'pattern_b_test')
    r['weights'] = weights_norm
    weight_results.append(r)

    if r.get('spearman_corr', -999) > best_spearman:
        best_spearman = r['spearman_corr']
        best_weights  = weights_norm

df_weight_results = pd.DataFrame(weight_results)
print('=== ウェイト候補別スピアマン相関（学習データ）===')
print(df_weight_results[['weights', 'spearman_corr', 'win_top1_rate', 'show_top1_rate']]
      .to_string(index=False, float_format='{:.4f}'.format))
print(f'\n最適ウェイト（学習データ基準）: {tuple(round(w, 4) for w in best_weights)}')
print(f'学習 最大スピアマン相関: {best_spearman:.4f}')

In [ ]:
# === テストデータでの検証（最適ウェイトを適用）===
if len(df_lags_test) > 0:
    df_eval_test = df_lags_test[df_lags_test['finish_pos'].notna()].copy()
    df_eval_test['pattern_b_best'] = df_eval_test.apply(
        lambda r: calc_pattern_b(r, weights=best_weights), axis=1
    )

    test_results = {}
    for name, col in [('A (max5)', None), ('B (best weights)', 'pattern_b_best'), ('raw megu_index', 'megu_index')]:
        if col is None:
            df_eval_test['pattern_a'] = df_eval_test.apply(calc_pattern_a, axis=1)
            col = 'pattern_a'
        test_results[name] = eval_pattern(df_eval_test, col)

    df_test_result = pd.DataFrame(test_results).T

    print('=== テストデータ（2026年）での有効性検証 ===')
    print(df_test_result.to_string(float_format='{:.4f}'.format))

    print(f'\n=== 学習 vs テスト 汎化比較（パターンB最適ウェイト）===')
    train_sp = eval_pattern(df_eval_train, 'pattern_b')['spearman_corr']
    test_sp  = eval_pattern(df_eval_test, 'pattern_b_best')['spearman_corr']
    print(f'  学習 スピアマン相関: {train_sp:.4f}')
    print(f'  テスト スピアマン相関: {test_sp:.4f}')
    if not np.isnan(test_sp) and not np.isnan(train_sp):
        ratio = test_sp / train_sp if train_sp != 0 else np.nan
        print(f'  汎化比率 (test/train): {ratio:.3f}  (1.0 = 過学習なし)')
else:
    print('テストデータなし（期間外）— スキップ')

## 4. 結論とウェイト確定

In [ ]:
print('=== パターンB 最終推奨ウェイト ===')
for i, (w, label) in enumerate(zip(best_weights, ['直近1走', '直近2走', '直近3走', '直近4走', '直近5走']), 1):
    print(f'  {label}: ×{w:.2f}')

print(f'\nデフォルト (0.35/0.25/0.20/0.12/0.08) との変更:')
default_w = (0.35, 0.25, 0.20, 0.12, 0.08)
default_sum = sum(default_w)
default_w_norm = tuple(w / default_sum for w in default_w)
for i, (dw, bw) in enumerate(zip(default_w_norm, best_weights), 1):
    diff = bw - dw
    marker = '→ 変更' if abs(diff) > 0.01 else ''
    print(f'  lag{i}: {dw:.3f} → {bw:.3f}  ({diff:+.3f}) {marker}')

# 確定値を保存
import json
final_weights = {
    'pattern_b_weights': list(best_weights),
    'best_spearman_train': float(best_spearman),
}

# テスト結果があれば追記
if len(df_lags_test) > 0 and 'df_eval_test' in dir():
    test_sp_best = eval_pattern(df_eval_test, 'pattern_b_best').get('spearman_corr', None)
    if test_sp_best is not None and not np.isnan(test_sp_best):
        final_weights['best_spearman_test'] = float(test_sp_best)
        final_weights['generalization_ratio'] = float(test_sp_best / best_spearman) if best_spearman != 0 else None

with open(OUTPUT_DIR / 'final_weights.json', 'w') as f:
    json.dump(final_weights, f, indent=2, ensure_ascii=False)

print(f'\n最終ウェイトを保存: {OUTPUT_DIR / "final_weights.json"}')

# 精度サマリーを保存
df_eval_result.to_parquet(OUTPUT_DIR / 'pattern_comparison.parquet')
df_weight_results.to_parquet(OUTPUT_DIR / 'weight_grid_search.parquet')
print('精度サマリー・グリッドサーチ結果を保存しました')